# ML benchmark

This section compares simple prediction benchmarks for the aggregate panel.

## Setup and data

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
panel = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'panel_skeleton.csv')
controls = [
    'gdp_per_capita_eur',
    'unemployment_rate_pc',
    'poverty_or_social_exclusion_pc',
    'government_health_expenditure_gdp_pc',
    'compulsory_health_financing_gdp_pc',
]
data = panel[['geo', 'year', 'unmet_need_pc'] + controls].dropna().copy()
data['log_gdp_per_capita'] = np.log(data['gdp_per_capita_eur'])
features = [
    'log_gdp_per_capita',
    'unemployment_rate_pc',
    'poverty_or_social_exclusion_pc',
    'government_health_expenditure_gdp_pc',
    'compulsory_health_financing_gdp_pc',
]
data[['geo', 'year', 'unmet_need_pc'] + features].head()

,geo,year,unmet_need_pc,log_gdp_per_capita,unemployment_rate_pc,poverty_or_social_exclusion_pc,government_health_expenditure_gdp_pc,compulsory_health_financing_gdp_pc
14,AT,2015,0.1,10.587594,6.1,16.9,8.3,7.74
15,AT,2016,0.2,10.613738,6.5,17.2,8.3,7.71
16,AT,2017,0.2,10.639694,5.9,17.1,8.3,7.74
17,AT,2018,0.1,10.677293,5.2,16.8,8.3,7.78
18,AT,2019,0.3,10.704816,4.8,16.5,8.4,7.91


## Train and test split

The split uses time order. Rows from 2015 to 2019 are used for training. Rows from 2020 to 2024 are held out for testing.

In [2]:
train = data.loc[data['year'] <= 2019].copy()
test = data.loc[data['year'] >= 2020].copy()
X_train = train[features]
y_train = train['unmet_need_pc']
X_test = test[features]
y_test = test['unmet_need_pc']
pd.DataFrame({
    'train_rows': [len(train)],
    'test_rows': [len(test)],
    'train_years': [f"{train['year'].min()}-{train['year'].max()}"],
    'test_years': [f"{test['year'].min()}-{test['year'].max()}"],
})

,train_rows,test_rows,train_years,test_years
0,150,132,2015-2019,2020-2024


## Baseline predictive models

In [3]:
def metrics(name, y_true, y_pred):
    return {
        'model': name,
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r_squared': r2_score(y_true, y_pred),
    }

performance = []

linear_model = Pipeline([
    ('scale', StandardScaler()),
    ('model', LinearRegression()),
])
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)
performance.append(metrics('Linear', y_test, linear_pred))

tree_model = GradientBoostingRegressor(
    n_estimators=150,
    learning_rate=0.04,
    max_depth=2,
    random_state=42,
)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
performance.append(metrics('Tree', y_test, tree_pred))

pd.DataFrame(performance).round(3)

,model,mae,rmse,r_squared
0,Linear,1.605,2.323,0.168
1,Tree,1.951,2.751,-0.167


## Small MLP model

In [4]:
torch.manual_seed(42)
np.random.seed(42)

# fit scaling on training rows only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype('float32')
X_test_scaled = scaler.transform(X_test).astype('float32')
y_train_array = y_train.to_numpy(dtype='float32').reshape(-1, 1)
y_test_array = y_test.to_numpy(dtype='float32').reshape(-1, 1)

validation_mask = train['year'].eq(2019).to_numpy()
X_inner = torch.tensor(X_train_scaled[~validation_mask])
y_inner = torch.tensor(y_train_array[~validation_mask])
X_valid = torch.tensor(X_train_scaled[validation_mask])
y_valid = torch.tensor(y_train_array[validation_mask])
X_test_tensor = torch.tensor(X_test_scaled)

mlp = nn.Sequential(
    nn.Linear(len(features), 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.01, weight_decay=1e-4)

best_state = None
best_valid = float('inf')
patience = 30
wait = 0
for epoch in range(500):
    mlp.train()
    optimizer.zero_grad()
    train_loss = loss_fn(mlp(X_inner), y_inner)
    train_loss.backward()
    optimizer.step()

    mlp.eval()
    with torch.no_grad():
        valid_loss = loss_fn(mlp(X_valid), y_valid).item()
    if valid_loss < best_valid - 1e-5:
        best_valid = valid_loss
        best_state = {key: value.detach().clone() for key, value in mlp.state_dict().items()}
        wait = 0
    else:
        wait += 1
    if wait >= patience:
        break

if best_state is not None:
    mlp.load_state_dict(best_state)
mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(X_test_tensor).numpy().ravel()
performance.append(metrics('MLP', y_test, mlp_pred))

performance_table = pd.DataFrame(performance).round(4)
performance_table.to_csv(OUTPUTS_DIR / 'table_ml_performance.csv', index=False)
pd.DataFrame({'epochs_used': [epoch + 1], 'validation_rmse': [np.sqrt(best_valid)]})

,epochs_used,validation_rmse
0,223,1.983411


In [5]:
performance_table

,model,mae,rmse,r_squared
0,Linear,1.6052,2.3233,0.1681
1,Tree,1.9505,2.7514,-0.1668
2,MLP,2.5225,3.3476,-0.7272


## Brief ML summary

In [6]:
best_mae = performance_table.sort_values('mae').iloc[0]
best_rmse = performance_table.sort_values('rmse').iloc[0]
linear_row = performance_table.loc[performance_table['model'].eq('Linear')].iloc[0]
mae_gain = linear_row['mae'] - best_mae['mae']
rmse_gain = linear_row['rmse'] - best_rmse['rmse']
summary = f'''# ML benchmark summary

The benchmark uses {len(train)} training rows from 2015-2019 and {len(test)} test rows from 2020-2024.

The lowest test MAE is from the {best_mae['model']} model, with MAE {best_mae['mae']:.3f}. The lowest test RMSE is from the {best_rmse['model']} model, with RMSE {best_rmse['rmse']:.3f}.

Compared with the linear baseline, the best MAE is lower by {mae_gain:.3f} percentage points and the best RMSE is lower by {rmse_gain:.3f} percentage points.

These results describe predictive performance on aggregate Eurostat data. They do not support claims about cause and result or policy change.
'''
(OUTPUTS_DIR / 'ml_benchmark_summary.md').write_text(summary, encoding='utf-8')
summary

'# ML benchmark summary\n\nThe benchmark uses 150 training rows from 2015-2019 and 132 test rows from 2020-2024.\n\nThe lowest test MAE is from the Linear model, with MAE 1.605. The lowest test RMSE is from the Linear model, with RMSE 2.323.\n\nCompared with the linear baseline, the best MAE is lower by 0.000 percentage points and the best RMSE is lower by 0.000 percentage points.\n\nThese results describe predictive performance on aggregate Eurostat data. They do not support claims about cause and result or policy change.\n'